# Setup

## **Imports**

In [22]:
from pathlib import Path
import json
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from datetime import datetime


## **Paths and Device**

In [23]:
ROOT = Path(
    "/kaggle/input/competitions/filament-segmentation-2026/"
    "MAGFiLO_1.0_Kaggle_2026"
)

TRAIN_IMAGES = ROOT / "train" / "train_images"
TEST_IMAGES = ROOT / "test" / "test_images"

ANNOTATIONS_PATH = (
    ROOT
    / "train"
    / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
)

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

gpu_count = torch.cuda.device_count()

print("Device:", device)
print("Available GPUs:", gpu_count)

if torch.cuda.is_available():
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

Device: cuda
Available GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## **Model Utilities**

In [24]:
def prepare_model(model):
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)

    return model.to(device)

# Data
## Load Annotations

In [25]:
with open(ANNOTATIONS_PATH, "r") as f:
    annotations_data = json.load(f)

## Data Utilities

In [26]:
def index_by(items, key):
    index = {}

    for item in items:
        value = item[key]
        index.setdefault(value, []).append(item)

    return index


image_entries_by_filename = index_by(
    annotations_data["images"],
    "file_name"
)

annotations_by_image_id = index_by(
    annotations_data["annotations"],
    "image_id"
)


def load_image(filename):
    return Image.open(
        TRAIN_IMAGES / filename
    ).convert("L")


def get_annotated_filenames():
    return sorted(
        image_entries_by_filename.keys()
    )


def get_image_entries(filename):
    return image_entries_by_filename.get(
        filename,
        []
    )


def get_annotations(image_id):
    return annotations_by_image_id.get(
        image_id,
        []
    )


def get_image_annotations(
    filename,
    annotation_set=0
):
    image_entries = get_image_entries(filename)

    if not image_entries:
        raise ValueError(
            f"No annotations found for {filename}"
        )

    if not 0 <= annotation_set < len(image_entries):
        raise ValueError(
            f"Annotation set {annotation_set} does not exist "
            f"for {filename}. "
            f"Available sets: {len(image_entries)}"
        )

    image_id = image_entries[annotation_set]["id"]

    return get_annotations(image_id)


def annotations_to_mask(
    annotations,
    image_size
):
    mask = Image.new(
        "L",
        image_size,
        0
    )

    draw = ImageDraw.Draw(mask)

    for annotation in annotations:
        segmentation = annotation["segmentation"][0]

        points = [
            (
                segmentation[i],
                segmentation[i + 1]
            )
            for i in range(
                0,
                len(segmentation),
                2
            )
        ]

        draw.polygon(
            points,
            fill=1
        )

    return np.array(
        mask,
        dtype=np.uint8
    )


def get_image_and_mask(
    filename,
    annotation_set=0
):
    image = load_image(filename)

    annotations = get_image_annotations(
        filename,
        annotation_set
    )

    mask = annotations_to_mask(
        annotations,
        image.size
    )

    return image, mask


def get_training_samples():
    samples = []

    for filename, image_entries in (
        image_entries_by_filename.items()
    ):
        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return sorted(samples)

## Dataset

In [27]:
class FilamentDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        filename, annotation_set = self.samples[index]

        image, mask = get_image_and_mask(
            filename,
            annotation_set
        )

        image = np.array(
            image,
            dtype=np.float32
        ) / 255.0

        mask = mask.astype(np.float32)

        image = torch.from_numpy(
            image
        ).unsqueeze(0)

        mask = torch.from_numpy(
            mask
        ).unsqueeze(0)

        return image, mask

## Data Splitting

**Splitting Utilities**

In [28]:
def get_observation_date(filename):
    timestamp = filename[:14]

    return datetime.strptime(
        timestamp,
        "%Y%m%d%H%M%S"
    ).date()


def group_filenames_by_date(filenames):
    filenames_by_date = {}

    for filename in filenames:
        date = get_observation_date(filename)

        filenames_by_date.setdefault(
            date,
            []
        ).append(filename)

    return filenames_by_date


def split_filenames_by_date(
    filenames,
    validation_fraction=0.2,
    seed=42
):
    filenames_by_date = group_filenames_by_date(
        filenames
    )

    dates = np.array(
        sorted(filenames_by_date.keys())
    )

    rng = np.random.default_rng(seed)
    rng.shuffle(dates)

    validation_count = max(
        1,
        round(len(dates) * validation_fraction)
    )

    validation_dates = set(
        dates[:validation_count]
    )

    train_filenames = []
    validation_filenames = []

    for date, date_filenames in filenames_by_date.items():
        if date in validation_dates:
            validation_filenames.extend(date_filenames)
        else:
            train_filenames.extend(date_filenames)

    return (
        sorted(train_filenames),
        sorted(validation_filenames)
    )


def get_samples_for_filenames(filenames):
    samples = []

    for filename in filenames:
        image_entries = get_image_entries(filename)

        for annotation_set in range(
            len(image_entries)
        ):
            samples.append(
                (filename, annotation_set)
            )

    return samples

**Split Script**

In [29]:
filenames = get_annotated_filenames()

train_filenames, validation_filenames = (
    split_filenames_by_date(filenames)
)

train_samples = get_samples_for_filenames(
    train_filenames
)

validation_samples = get_samples_for_filenames(
    validation_filenames
)

print("Training images:", len(train_filenames))
print("Validation images:", len(validation_filenames))

print("Training samples:", len(train_samples))
print("Validation samples:", len(validation_samples))

Training images: 562
Validation images: 145
Training samples: 905
Validation samples: 249


## DataLoaders

In [30]:
train_dataset = FilamentDataset(train_samples)
validation_dataset = FilamentDataset(validation_samples)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

images, mask = next(iter(train_loader))

print("Train batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("Image batch shape:", images.shape)
print("Mask batch shape:", mask.shape)

Train batches: 452
Validation batches: 125
Image batch shape: torch.Size([2, 1, 2048, 2048])
Mask batch shape: torch.Size([2, 1, 2048, 2048])


# Training
## Losses

In [31]:
def dice_loss(logits, targets, eps=1e-6):
    probabilities = torch.sigmoid(logits)

    intersection = (
        probabilities * targets
    ).sum(dim=(1, 2, 3))

    total = (
        probabilities.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return 1 - dice.mean()


def segmentation_loss(logits, targets):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets
    )

    dice = dice_loss(
        logits,
        targets
    )

    return bce + dice

## Metrics

In [32]:
def dice_score(logits, targets, threshold=0.5, eps=1e-6):
    predictions = (
        torch.sigmoid(logits) >= threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    total = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
    )

    dice = (
        2 * intersection + eps
    ) / (
        total + eps
    )

    return dice.mean().item()


def iou_score(logits, targets, threshold=0.5, eps=1e-6):
    predictions = (
        torch.sigmoid(logits) >= threshold
    ).float()

    intersection = (
        predictions * targets
    ).sum(dim=(1, 2, 3))

    union = (
        predictions.sum(dim=(1, 2, 3))
        + targets.sum(dim=(1, 2, 3))
        - intersection
    )

    iou = (
        intersection + eps
    ) / (
        union + eps
    )

    return iou.mean().item()

## Training Utilities

In [33]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    device,
    scaler
):
    model.train()

    total_loss = 0.0

    for images, masks in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)
            loss = criterion(
                logits,
                masks
            )

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(
            device,
            non_blocking=True
        )

        masks = masks.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)

            loss = criterion(
                logits,
                masks
            )

        total_loss += loss.item()

        total_dice += dice_score(
            logits,
            masks
        )

        total_iou += iou_score(
            logits,
            masks
        )

    batches = len(loader)

    return {
        "loss": total_loss / batches,
        "dice": total_dice / batches,
        "iou": total_iou / batches
    }


def train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    epochs,
    checkpoint_path,
    patience=3
):
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )

    history = {
        "train_loss": [],
        "validation_loss": [],
        "dice": [],
        "iou": []
    }

    best_validation_loss = float("inf")
    epochs_without_improvement = 0

    start_time = time.time()

    for epoch in range(epochs):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
            scaler=scaler
        )

        validation = validate_one_epoch(
            model=model,
            loader=validation_loader,
            criterion=criterion,
            device=device
        )

        history["train_loss"].append(train_loss)
        history["validation_loss"].append(
            validation["loss"]
        )
        history["dice"].append(
            validation["dice"]
        )
        history["iou"].append(
            validation["iou"]
        )

        print(
            f"Epoch {epoch + 1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {validation['loss']:.4f} | "
            f"Dice: {validation['dice']:.4f} | "
            f"IoU: {validation['iou']:.4f}"
        )

        if validation["loss"] < best_validation_loss:
            best_validation_loss = validation["loss"]
            epochs_without_improvement = 0

            model_to_save = (
                model.module
                if isinstance(model, nn.DataParallel)
                else model
            )

            torch.save(
                model_to_save.state_dict(),
                checkpoint_path
            )

            print("  Saved new best model.")

        else:
            epochs_without_improvement += 1

            print(
                f"  No improvement "
                f"({epochs_without_improvement}/{patience})"
            )

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

    elapsed = time.time() - start_time

    print(
        f"Training time: "
        f"{elapsed / 60:.1f} minutes"
    )

    return history

# Backbones

In [34]:
!pip install -q segmentation-models-pytorch

In [35]:
import segmentation_models_pytorch as smp

## U-Net++

In [36]:
def make_unetplusplus():
    return smp.UnetPlusPlus(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1
    )

## DeepLabV3+

In [37]:
def make_deeplabv3plus():
    return smp.DeepLabV3Plus(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1
    )

## SegFormer

In [38]:
def make_segformer():
    return smp.Segformer(
        encoder_name="mit_b0",
        encoder_weights="imagenet",
        in_channels=1,
        classes=1
    )

# Experiments
## Experiment Utilities

In [39]:
def run_experiment(
    name,
    model,
    epochs=20,
    learning_rate=1e-4,
    patience=3
):
    checkpoint_path = (
        CHECKPOINT_DIR / f"{name}.pt"
    )

    # ---------- Setup ----------

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    model = prepare_model(model)

    print(f"Experiment: {name}")
    print(f"Parameters: {parameter_count:,}")
    print(f"Checkpoint: {checkpoint_path}")

    # ---------- Sanity Check ----------

    images, masks = next(iter(train_loader))

    images = images.to(
        device,
        non_blocking=True
    )

    masks = masks.to(
        device,
        non_blocking=True
    )

    model.eval()

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            logits = model(images)

    # CUDA operations are asynchronous.
    # Force the forward pass to finish here so any CUDA
    # error is reported at the sanity check rather than later.
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    if logits.shape != masks.shape:
        raise ValueError(
            f"Model output shape {logits.shape} "
            f"does not match mask shape {masks.shape}."
        )

    print(f"Input shape: {images.shape}")
    print(f"Output shape: {logits.shape}")

    if torch.cuda.is_available():
        peak_memory_gb = (
            torch.cuda.max_memory_allocated()
            / 1024**3
        )

        print(
            f"Peak GPU 0 memory during sanity check: "
            f"{peak_memory_gb:.2f} GB"
        )

    del images, masks, logits

    # ---------- Training ----------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate
    )

    history = train_model(
        model=model,
        train_loader=train_loader,
        validation_loader=validation_loader,
        optimizer=optimizer,
        criterion=segmentation_loss,
        device=device,
        epochs=epochs,
        checkpoint_path=checkpoint_path,
        patience=patience
    )

    return {
        "name": name,
        "history": history,
        "checkpoint_path": checkpoint_path,
        "parameters": parameter_count
    }

## U-Net++ Baseline

In [40]:
# unetplusplus_result = run_experiment(
#     name="unetplusplus_resnet34",
#     model=make_unetplusplus(),
#     epochs=5,
#     learning_rate=1e-4,
#     patience=2
# )

## DeepLabV3+ Baseline

In [41]:
# deeplabv3plus_result = run_experiment(
#     name="deeplabv3plus_resnet34",
#     model=make_deeplabv3plus(),
#     epochs=5,
#     learning_rate=1e-4,
#     patience=2
# )

## SegFormer Baseline

In [42]:
segformer_result = run_experiment(
    name="segformer_mit_b1",
    model=make_segformer(),
    epochs=5,
    learning_rate=1e-4,
    patience=2
)

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.3M [00:00<?, ?B/s]

Experiment: segformer_mit_b1
Parameters: 3,711,265
Checkpoint: /kaggle/working/checkpoints/segformer_mit_b1.pt
Input shape: torch.Size([2, 1, 2048, 2048])
Output shape: torch.Size([2, 1, 2048, 2048])
Peak GPU 0 memory during sanity check: 6.33 GB


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/segmentation_models_pytorch/base/model.py", line 66, in forward
    features = self.encoder(x)
               ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwa